# 🧠 AI Research Foundations - Module 07: Autograd & Neural Networks From Scratch

Welcome to Module 07! Run this notebook interactively in **Google Colab** with zero installation.

In this notebook, we:
1. Build a scalar autograd engine (`Value` class)
2. Implement reverse-mode automatic differentiation with topological sort
3. Assemble an MLP and train it to solve the non-linear XOR problem

In [ ]:
import random
import matplotlib.pyplot as plt

class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "ReLU")
        def _backward():
            self.grad += (out.data > 0.0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

## 1. Verifying Autograd with Analytical Calculus
Let $f(x, y) = (x \cdot y + 2)^2$. At $x=3, y=-4$:
- $f = (3 \cdot (-4) + 2)^2 = (-10)^2 = 100$
- $\frac{df}{dx} = 2(x \cdot y + 2) \cdot y = 2(-10)(-4) = 80$
- $\frac{df}{dy} = 2(x \cdot y + 2) \cdot x = 2(-10)(3) = -60$

In [ ]:
x = Value(3.0)
y = Value(-4.0)
z = x * y + 2.0
f = z * z
f.backward()

print(f"Function value f: {f.data:.1f} (Expected: 100.0)")
print(f"Computed df/dx: {x.grad:.1f} (Expected: 80.0)")
print(f"Computed df/dy: {y.grad:.1f} (Expected: -60.0)")
assert f.data == 100.0 and x.grad == 80.0 and y.grad == -60.0
print("✅ Autograd matches analytical calculus exactly!")

## 2. Solving XOR with a Custom MLP

In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.relu()
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    def __call__(self, x):
        return [n(x) for n in self.neurons]
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

random.seed(42)
xs = [[Value(0), Value(0)], [Value(0), Value(1)], [Value(1), Value(0)], [Value(1), Value(1)]]
ys = [0.0, 1.0, 1.0, 0.0]

model = MLP(2, [4, 4, 1])
losses = []

for epoch in range(120):
    ypred = [model(x) for x in xs]
    loss = sum((yout - Value(ygt))**2 for ygt, yout in zip(ys, ypred))
    
    for p in model.parameters():
        p.grad = 0.0
    loss.backward()
    for p in model.parameters():
        p.data -= 0.1 * p.grad
    losses.append(loss.data)

plt.figure(figsize=(6, 3.5), dpi=100)
plt.plot(losses, color="#3182CE", lw=2)
plt.title("Micrograd MLP Training Loss on XOR", fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Squared Error Loss")
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

for x, ygt in zip(xs, ys):
    print(f"XOR Input: {[int(xi.data) for xi in x]} | Target: {int(ygt)} | Prediction: {model(x).data:.3f}")
print("\n✅ Module 07 notebook completed successfully!")